In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, max, min, avg, when
spark = SparkSession.builder.appName('1_2').getOrCreate()

In [106]:
class aggregator:
    def __init__(self):
        self._input_csv = spark.read \
                        .option('header', True) \
                        .option('inferSchema', True) \
                        .csv('input.csv')

    def query(self, ticker, start, end):
        filtered_df = self._input_csv \
                            .filter(col('symbol') == ticker) \
                            .filter(to_date(col('timestamp'))>= start) \
                            .filter(to_date(col('timestamp'))<=end)

        
        
        
        aggregate_df = filtered_df.withColumn('date', to_date('timestamp')) \
                        .groupBy('symbol', 'date') \
                        .agg(
                            max(col('price')).alias('max'),
                            max(col('timestamp')).alias('max_timestamp'),
                            min(col('price')).alias('min'),
                            min(col('timestamp')).alias('min_timestamp'),
                            avg(col('price')).alias('avg')
                        ).withColumnRenamed('symbol', 'agg_symbol')
        
        open_df = aggregate_df.join(filtered_df, (aggregate_df.agg_symbol == filtered_df.symbol) 
                                    & (aggregate_df.min_timestamp == filtered_df.timestamp), how='inner') \
                                .select('symbol', 'timestamp', 'price')

        close_df = aggregate_df.join(filtered_df, (aggregate_df.agg_symbol == filtered_df.symbol) 
                                    & (aggregate_df.max_timestamp == filtered_df.timestamp), how='inner') \
                                .select('symbol', 'timestamp', 'price')
        
        final_df1 = aggregate_df.join(open_df, (aggregate_df.agg_symbol == open_df.symbol) & 
                                     (aggregate_df.date == to_date(open_df.timestamp)), how='inner') \
                                .withColumnRenamed('price', 'open') \
                                .select('agg_symbol', 'date', 'max', 'min', 'avg', 'open')

        final_df = final_df1.join(close_df, (final_df1.agg_symbol == close_df.symbol) & 
                                     (final_df1.date == to_date(close_df.timestamp)), how='inner') \
                                .withColumnRenamed('price', 'close') \
                                .select('agg_symbol', 'date', 'max', 'min', 'avg', 'open', 'close') \
                                .withColumnRenamed('agg_symbol', 'symbol')
                                
        
                                

        final_df.show()
        


In [107]:
obj = aggregator()
obj.query('AAPL', '2024-01-01', '2025-01-01')

+------+----------+-----+-----+------+-----+-----+
|symbol|      date|  max|  min|   avg| open|close|
+------+----------+-----+-----+------+-----+-----+
|  AAPL|2024-01-01|186.5|184.8| 185.5|185.2|184.8|
|  AAPL|2024-01-02|188.0|187.1|187.55|187.1|188.0|
+------+----------+-----+-----+------+-----+-----+

